In [4]:
# ChromaDB 하려면 설치하고 써야 함
# %pip install chromadb llama_index.vector_stores.chroma llama-index-llms-ollama llama-index-embeddings-ollama

In [5]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [6]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model = 'gemma2:2b',
    temperature = 0.5, # 생성되는 텍스트의 다양성을 조절하는 매개변수
    request_timeout = 120.0 # 요청이 타임아웃되기까지의 시간(초)_이 시간이 되면 멈춰라
)

embed_model = OllamaEmbedding(
    model_name = 'nomic-embed-text'
)

In [7]:
# 벡터 DB 생성 및 저장
db = chromadb.PersistentClient(path="./chroma_db") 

# 컬렉션 생성 및 저장
chroma_collection = db.get_or_create_collection("quickstart_ollama") # 스키마 정의

In [8]:
# ChromaDB를 LlamaIndex의 인덱싱 및 검색 파이프라인에 통합

vector_store = ChromaVectorStore(chroma_collection=chroma_collection) # ChromaVectorStore 생성
storage_context = StorageContext.from_defaults(vector_store=vector_store) # 스토리지 컨텍스트 생성
# 연결할 준비를 한 것

In [13]:
# 저장된 벡터로 부터 인덱스 로드
index = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True
)

---
### 메모리 로드

In [14]:
# 쿼리 엔진
query_engine = index.as_query_engine(llm = llm)

In [15]:
# 쿼리 실행
query = "이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요."
response = query_engine.query(query)
# 응답 출력
print("\n질문: ", query)
print("답변: ", response)
# 더 자세한 답변을 원하면 파라메터 중 temperature 값을 높이면 된다 


질문:  이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요.
답변:  이 논문에서는 Transformer 모델이 기존 모델보다  BLEU 점수를 높이고, 더 빠른 학습 시간으로 효율적인 모델임을 보여주고 있습니다. 특히,  Transformer 모델은  "새로운 상황에 맞춰서 변화하는 정보를 처리할 수 있음"과 "더 많은 정보를 사용하면서도 비용을 줄이는 것을 가능하게 함" 등의 장점을 가지고 있다는 점이 강조됩니다. 

